# Multi-Query RAG
### Expanding one query into several to widen recall

Corpus: `OWASP Top 10 for LLM Applications (2025)` — 10 named risk categories (LLM01–LLM10) sharing vocabulary like “risk”, “attack”, “model”, which is exactly what makes naive retrieval struggle.

## Step 1: Build the pipeline

In [8]:
import pikepdf
pikepdf.open("OWASP-Top-10-for-LLMs-v2025.pdf").save("OWASP-Top-10-for-LLMs-v2025-fixed.pdf")

In [9]:
!pip install langchain langchain-community langchain-ollama langchain-text-splitters faiss-cpu pypdf -q

^C



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings, ChatOllama


In [11]:
PDF_PATH = "OWASP-Top-10-for-LLMs-v2025.pdf"

pages = PyPDFLoader(PDF_PATH).load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(pages)

embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")
vector_store = FAISS.from_documents(chunks, embeddings)

llm = ChatOllama(model="llama3.2:3b", temperature=0)

print(f"Loaded {len(pages)} pages -> {len(chunks)} chunks -> {vector_store.index.ntotal} vectors")

incorrect startxref pointer(1)
parsing for Object Streams
Error -3 while decompressing data: invalid code lengths set
Error -3 while decompressing data: invalid code lengths set
Error -3 while decompressing data: invalid code lengths set
Error -3 while decompressing data: invalid code lengths set


Loaded 45 pages -> 131 chunks -> 131 vectors


## Step 2: Baseline — a single broad query
A broad question tends to collapse onto whichever single region of embedding space is closest — usually just one category.

In [4]:
query = "How can an LLM application be attacked?"

baseline_docs = vector_store.similarity_search(query, k=4)
print("Single query — retrieved chunks:")
for doc in baseline_docs:
    print(f"page {doc.metadata['page']}: {doc.page_content[:120]}...")

baseline_pages = {doc.metadata["page"] for doc in baseline_docs}
print(f"\nUnique pages touched: {sorted(baseline_pages)}")

Single query — retrieved chunks:
page 8: An attacker modifies a document in a repository used by a Retrieval-Augmented Generation
(RAG) application. When a user'...
page 29: guardrails bypass, improper separation of privileges, etc. Even if the exact wording is not
disclosed, attackers interac...
page 17: frameworks like vLLM or OpenLLM. The compromised LoRA adapter is subtly altered to
include hidden vulnerabilities and ma...
page 8: Example Attack Scenarios
Scenario #1: Direct Injection
An attacker injects a prompt into a customer support chatbot, ins...

Unique pages touched: [8, 17, 29]


## Step 3: LLM generates query variations
Each variation should use different vocabulary or focus on a different angle, so each one activates a different region of embedding space.

In [ ]:
expand_prompt = """Generate 4 different ways to ask the following question. Use different vocabulary
and focus on a different angle or category each time. Return exactly one question per line, no numbering.

Question: {query}"""

variations_text = llm.invoke(expand_prompt.format(query=query)).content.strip()
variations = [v.strip("-* ").strip() for v in variations_text.split("\n") if v.strip()]

print(f"Generated 4 variations:")
for v in variations:
    print(f"  - {v}")


Generated 5 variations:
  - Here are four different ways to ask the question:
  - How might an adversary exploit the vulnerabilities of a large language model?
  - What weaknesses could an attacker target in order to compromise the performance or security of a language-based AI system?
  - In what ways might a malicious actor attempt to subvert the goals or intentions of a language model application?
  - Can an LLM be vulnerable to attacks that manipulate its understanding of context, semantics, or syntax?


## Step 4: Retrieve per variation, then merge and deduplicate

In [6]:
merged = {}
for q in [query] + variations:
    for doc in vector_store.similarity_search(q, k=4):
        key = (doc.metadata["page"], doc.page_content[:60])
        merged[key] = doc

merged_pages = {doc.metadata["page"] for doc in merged.values()}
print(f"Single query:  {len(baseline_docs)} chunks, {len(baseline_pages)} unique pages -> {sorted(baseline_pages)}")
print(f"Multi-query:   {len(merged)} unique chunks, {len(merged_pages)} unique pages -> {sorted(merged_pages)}")


Single query:  4 chunks, 3 unique pages -> [8, 17, 29]
Multi-query:   15 unique chunks, 13 unique pages -> [8, 10, 15, 17, 19, 20, 22, 23, 29, 34, 37, 38, 39]


## Step 5: Generate the final answer from the merged context

In [7]:
context = "\n\n".join(doc.page_content for doc in merged.values())

prompt = f"""Answer the question based only on the following context. Cover every distinct risk mentioned.

{context}

Question: {query}
Answer:"""

print(llm.invoke(prompt).content)

An LLM (Large Language Model) application can be attacked in various ways, including:

1. **Direct Injection**: An attacker injects a malicious prompt into the LLM, which alters its output and generates misleading results.
2. **Indirect Injection**: A user employs an LLM to summarize a webpage containing hidden instructions that cause the LLM to insert an image linking to a URL, leading to exfiltration of sensitive information.
3. **Unintentional Injection**: A company includes an instruction in a job description to identify AI-generated applications, and an applicant unknowingly triggers the AI detection by using an LLM to optimize their resume.
4. **Intentional Model Influence**: An attacker modifies a document in a repository used by a Retrieval-Augmented Generation (RAG) application, altering the LLM's output and generating misleading results.
5. **Payload Splitting**: An attacker uploads a resume with split malicious prompts, which are combined during evaluation to manipulate the 

## Try it yourself
1. Raise the variation count to 6-8 and see how much further page coverage grows.
2. Try a narrow, specific query — multi-query should help far less than it did here.
3. Weight votes by how many variations retrieved the same chunk, instead of a plain union.